# 05 — Full Statistical Assessment Suite

This notebook consolidates **all** model evaluation results into a single master table.

**Assessment methods:**
1. VaR exceedance counting with binomial $p$-values at 95% and 99%
2. CVaR (Expected Shortfall) — required by Basel III
3. PIT-based QQ against Uniform(0,1) — per-window
4. Kolmogorov-Smirnov on PIT values against Uniform(0,1) 
5. Anderson-Darling on PIT values (tail-weighted)
6. Christoffersen independence test on hit sequences

All tests use the **corrected** per-window methodology: NIG, Student-T, and Gaussian distributions fitted (or evaluated) per rolling window.

In [1]:
# Imports

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
from scipy import stats
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "iframe"

import warnings
warnings.filterwarnings("ignore")

from src.data_utils import load_processed, save_processed
from src.nig import NIGParams, nig_pdf, nig_cdf
from src.assessment import (
    count_exceedances, binomial_pvalue, pvalue_color,
    christoffersen_test,
    pit_qq, pit_ks_test, anderson_darling_pit,
)

print("Imports OK")

Imports OK


In [2]:
# Load all data

innovations  = load_processed("innovations.parquet")
var99        = load_processed("var99.parquet")
var95        = load_processed("var95.parquet")
cvar99       = load_processed("cvar99.parquet")
cvar95       = load_processed("cvar95.parquet")
pit_nig_df   = load_processed("pit_nig.parquet")
pit_t_df     = load_processed("pit_t.parquet")
pit_gauss_df = load_processed("pit_gauss.parquet")
exceedance_df = load_processed("exceedance_results.parquet")

TICKERS = list(innovations.columns)

# Load full rolling results for actual returns
all_results = {}
for ticker in TICKERS:
    safe = ticker.replace("^", "").replace("=", "_")
    all_results[ticker] = load_processed(f"rolling_{safe}.parquet")

N = len(var99)
print(f"Loaded: {len(TICKERS)} assets, {N} predictions")
print(f"Date range: {var99.index[0].date()} → {var99.index[-1].date()}")

Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\innovations.parquet  shape=(1000, 5)
Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\var99.parquet  shape=(1000, 5)
Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\var95.parquet  shape=(1000, 5)
Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\cvar99.parquet  shape=(1000, 5)
Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-

---
## 1. KS and Anderson-Darling on PIT Values

All three PIT series (NIG, Student-T, Gaussian) were computed **per-window** in notebook 02. KS and AD test whether the PIT values are $U[0,1]$.

In [3]:
# KS test on PIT values: NIG vs Student-T vs Gaussian

ks_rows = []
for ticker in TICKERS:
    pit_n = pit_nig_df[ticker].dropna().values
    pit_t = pit_t_df[ticker].dropna().values
    pit_g = pit_gauss_df[ticker].dropna().values

    ks_n = pit_ks_test(pit_n)
    ks_t = pit_ks_test(pit_t)
    ks_g = pit_ks_test(pit_g)

    ks_nig_vs_gauss = (1 - ks_n["statistic"] / max(ks_g["statistic"], 1e-10)) * 100

    ks_rows.append({
        "Ticker": ticker,
        "NIG KS": ks_n["statistic"], "NIG p": ks_n["pvalue"],
        "T KS": ks_t["statistic"], "T p": ks_t["pvalue"],
        "Gauss KS": ks_g["statistic"], "Gauss p": ks_g["pvalue"],
        "NIG vs Gauss": f"{ks_nig_vs_gauss:.1f}%",
    })

ks_df = pd.DataFrame(ks_rows).set_index("Ticker")
print("Kolmogorov-Smirnov on PIT Values (per-window)\n")
display(ks_df)

Kolmogorov-Smirnov on PIT Values (per-window)



,NIG KS,NIG p,T KS,T p,Gauss KS,Gauss p,NIG vs Gauss
Ticker,,,,,,,
AAPL,0.0164,0.9467,0.0517,0.0091,0.0490,0.0158,66.5%
EURUSD=X,0.0139,0.9887,0.0387,0.0981,0.0343,0.1847,59.5%
GLD,0.0227,0.6726,0.0402,0.0764,0.0389,0.0949,41.6%
TIP,0.0295,0.3411,0.0423,0.0538,0.0368,0.1294,19.8%
^GSPC,0.0230,0.6563,0.0323,0.2426,0.0289,0.3667,20.4%


In [4]:
# Anderson-Darling on PIT values

ad_rows = []
for ticker in TICKERS:
    pit_n = pit_nig_df[ticker].dropna().values
    pit_t = pit_t_df[ticker].dropna().values
    pit_g = pit_gauss_df[ticker].dropna().values

    ad_n = anderson_darling_pit(pit_n)
    ad_t = anderson_darling_pit(pit_t)
    ad_g = anderson_darling_pit(pit_g)

    ad_vals = {"NIG": ad_n, "T": ad_t, "Gauss": ad_g}
    best = min(ad_vals, key=ad_vals.get)

    ad_rows.append({
        "Ticker": ticker,
        "NIG AD": round(ad_n, 4),
        "T AD": round(ad_t, 4),
        "Gauss AD": round(ad_g, 4),
        "Best (AD)": best,
    })

ad_df = pd.DataFrame(ad_rows).set_index("Ticker")
print("Anderson-Darling on PIT Values (tail-weighted)\n")
display(ad_df)
print("\nSmaller AD = better tail fit. AD gives extra weight to tail deviations.")

Anderson-Darling on PIT Values (tail-weighted)



,NIG AD,T AD,Gauss AD,Best (AD)
Ticker,,,,
AAPL,0.5076,3.6556,3.4028,NIG
EURUSD=X,0.3629,1.6492,1.4703,NIG
GLD,1.0998,1.9048,2.1005,NIG
TIP,0.8295,3.5030,2.6118,NIG
^GSPC,0.9754,2.5638,2.6287,NIG



Smaller AD = better tail fit. AD gives extra weight to tail deviations.


---
## 2. Christoffersen Independence Test

In [5]:
# Christoffersen test on VaR hit sequences

christ_rows = []
for ticker in TICKERS:
    actual = all_results[ticker]["return"].values
    for level, var_df in [(0.99, var99), (0.95, var95)]:
        var_s  = var_df[ticker].values
        hits   = (actual < var_s).astype(int)
        result = christoffersen_test(hits)

        christ_rows.append({
            "Ticker": ticker,
            "VaR level": f"{int(level*100)}%",
            "Statistic": result["statistic"],
            "p-value": result["pvalue"],
            "Independent": "✓" if result["independent"] else "✗",
        })

christ_df = pd.DataFrame(christ_rows)
print("Christoffersen Independence Test\n")
print(christ_df.to_string(index=False))
print("\n✓ = exceedances are independent (p > 0.05)")
print("✗ = exceedances cluster — residual volatility dynamics")

Christoffersen Independence Test

  Ticker VaR level  Statistic  p-value Independent
    AAPL       99%     6.1763   0.0129           ✗
    AAPL       95%     0.6662   0.4144           ✓
EURUSD=X       99%     0.2918   0.5891           ✓
EURUSD=X       95%     5.7135   0.0168           ✗
     GLD       99%     0.2918   0.5891           ✓
     GLD       95%     2.3766   0.1232           ✓
     TIP       99%     0.3980   0.5281           ✓
     TIP       95%     0.1037   0.7475           ✓
   ^GSPC       99%     1.1211   0.2897           ✓
   ^GSPC       95%     4.6932   0.0303           ✗

✓ = exceedances are independent (p > 0.05)
✗ = exceedances cluster — residual volatility dynamics


---
## 3. Master Assessment Table

Consolidates all test results into a single table per asset.

In [6]:
# Build master assessment table

# Reload exceedance results from notebook 04
exc_df = load_processed("exceedance_results.parquet")

master_rows = []
for ticker in TICKERS:
    actual = all_results[ticker]["return"].values

    # VaR exceedances
    exc99 = exc_df[(exc_df["Ticker"] == ticker) & (exc_df["VaR level"] == "99%")].iloc[0]
    exc95 = exc_df[(exc_df["Ticker"] == ticker) & (exc_df["VaR level"] == "95%")].iloc[0]

    # KS on PIT
    ks_nig = pit_ks_test(pit_nig_df[ticker].dropna().values)

    # AD on PIT
    ad_nig = anderson_darling_pit(pit_nig_df[ticker].dropna().values)

    # Christoffersen
    hits99 = (actual < var99[ticker].values).astype(int)
    hits95 = (actual < var95[ticker].values).astype(int)
    chr99 = christoffersen_test(hits99)
    chr95 = christoffersen_test(hits95)

    master_rows.append({
        "Ticker": ticker,
        "VaR99 exceed": exc99["Exceedances"],
        "VaR99 p-val": exc99["p-value"],
        "VaR99 result": exc99["Result"],
        "VaR95 exceed": exc95["Exceedances"],
        "VaR95 p-val": exc95["p-value"],
        "VaR95 result": exc95["Result"],
        "KS (NIG)": ks_nig["statistic"],
        "KS p": ks_nig["pvalue"],
        "AD (NIG)": round(ad_nig, 4),
        "Christ 99% p": chr99["pvalue"],
        "Christ 95% p": chr95["pvalue"],
    })

master_df = pd.DataFrame(master_rows).set_index("Ticker")
print("Master Assessment Table\n")
display(master_df)

Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\exceedance_results.parquet  shape=(10, 7)
Master Assessment Table



,VaR99 exceed,VaR99 p-val,VaR99 result,VaR95 exceed,VaR95 p-val,VaR95 result,KS (NIG),KS p,AD (NIG),Christ 99% p,Christ 95% p
Ticker,,,,,,,,,,,
AAPL,14,0.2006,GREEN,59,0.1912,GREEN,0.0164,0.9467,0.5076,0.0129,0.4144
EURUSD=X,12,0.5215,GREEN,52,0.7714,GREEN,0.0139,0.9887,0.3629,0.5891,0.0168
GLD,12,0.5215,GREEN,56,0.3834,GREEN,0.0227,0.6726,1.0998,0.5891,0.1232
TIP,14,0.2006,GREEN,50,1.0000,GREEN,0.0295,0.3411,0.8295,0.5281,0.7475
^GSPC,17,0.0365,RED,60,0.1465,GREEN,0.0230,0.6563,0.9754,0.2897,0.0303


In [7]:
# Color-coded master table

def result_color(val):
    if val == "GREEN": return "#2dc653"
    elif val == "RED": return "#e63946"
    elif val == "BLUE": return "#457b9d"
    return "#f8f9fa"

col_headers = [
    "Ticker", "VaR99<br>Exceed", "VaR99<br>p-val", "VaR99<br>Result",
    "VaR95<br>Exceed", "VaR95<br>p-val", "VaR95<br>Result",
    "KS (NIG)", "KS p", "AD (NIG)",
    "Christ<br>99% p", "Christ<br>95% p",
]

cell_values = [
    master_df.index.tolist(),
    master_df["VaR99 exceed"].tolist(),
    master_df["VaR99 p-val"].tolist(),
    master_df["VaR99 result"].tolist(),
    master_df["VaR95 exceed"].tolist(),
    master_df["VaR95 p-val"].tolist(),
    master_df["VaR95 result"].tolist(),
    master_df["KS (NIG)"].tolist(),
    master_df["KS p"].tolist(),
    master_df["AD (NIG)"].tolist(),
    master_df["Christ 99% p"].tolist(),
    master_df["Christ 95% p"].tolist(),
]

n_rows = len(master_df)
n_cols = len(col_headers)
default_fill = ["#f8f9fa"] * n_rows

fill_colors = [default_fill[:] for _ in range(n_cols)]
fill_colors[3] = [result_color(r) for r in master_df["VaR99 result"]]
fill_colors[6] = [result_color(r) for r in master_df["VaR95 result"]]

fig = go.Figure(data=[go.Table(
    header=dict(
        values=col_headers,
        fill_color="#2c3e50",
        font=dict(color="white", size=11),
        align="center", height=36,
    ),
    cells=dict(
        values=cell_values,
        fill_color=fill_colors,
        font=dict(size=11),
        align="center", height=28,
    ),
)])

fig.update_layout(
    title="Master Assessment Table — All Tests",
    template="plotly_white",
    height=320, margin=dict(t=50, b=10, l=10, r=10),
)
fig.write_html("../outputs/figures/05_master_table.html")
fig.show()
print("Master table saved")

Master table saved


---
## 4. Exceedance Timeline

In [8]:
# Cumulative exceedance timeline

colors_assets = dict(zip(TICKERS, px.colors.qualitative.Plotly))

fig = make_subplots(
    rows=len(TICKERS), cols=1,
    shared_xaxes=True,
    subplot_titles=TICKERS,
    vertical_spacing=0.06,
)

for i, ticker in enumerate(TICKERS):
    actual  = all_results[ticker]["return"]
    var99_s = var99[ticker]
    hits99  = (actual < var99_s).astype(int)
    cum_hits = hits99.cumsum()

    # Expected exceedances: linear line from 0 to N*0.01
    expected_line = np.linspace(0, N * 0.01, N)

    fig.add_trace(go.Scatter(
        x=actual.index, y=cum_hits.values,
        mode="lines", name=f"{ticker}",
        line=dict(color=colors_assets[ticker], width=1.5),
        showlegend=True,
    ), row=i+1, col=1)

    fig.add_trace(go.Scatter(
        x=actual.index, y=expected_line,
        mode="lines", name="Expected" if i == 0 else None,
        showlegend=(i == 0),
        line=dict(color="grey", width=1, dash="dash"),
    ), row=i+1, col=1)

    fig.update_yaxes(title_text="Cum. exceed.", row=i+1, col=1)

fig.update_layout(
    title="Cumulative VaR 99% Exceedances vs Expected",
    height=900, template="plotly_white",
    hovermode="x unified",
)
fig.write_html("../outputs/figures/05_exceedance_timeline.html")
fig.show()
print("Figure saved")

Figure saved


---
## 5. Model Verdict

In [9]:
# Final model verdict

print("="*60)
print("MODEL ASSESSMENT VERDICT")
print("="*60)

# VaR pass rates
var99_pass = (master_df["VaR99 result"] == "GREEN").sum()
var95_pass = (master_df["VaR95 result"] == "GREEN").sum()
n_assets = len(TICKERS)

print(f"\nVaR 99% exceedance: {var99_pass}/{n_assets} assets pass")
print(f"VaR 95% exceedance: {var95_pass}/{n_assets} assets pass")

# KS pass rate
ks_pass = (master_df["KS p"] > 0.05).sum()
print(f"KS test (NIG PIT):  {ks_pass}/{n_assets} assets pass")

# Christoffersen pass rate
chr99_pass = (master_df["Christ 99% p"] > 0.05).sum()
chr95_pass = (master_df["Christ 95% p"] > 0.05).sum()
print(f"Christoffersen 99%: {chr99_pass}/{n_assets} independent")
print(f"Christoffersen 95%: {chr95_pass}/{n_assets} independent")

# NIG vs Gaussian improvement
print(f"\nKS improvement (NIG vs Gaussian):")
for ticker in TICKERS:
    ks_n = pit_ks_test(pit_nig_df[ticker].dropna().values)["statistic"]
    ks_g = pit_ks_test(pit_gauss_df[ticker].dropna().values)["statistic"]
    imp = (1 - ks_n / max(ks_g, 1e-10)) * 100
    print(f"  {ticker:12s}  Gauss={ks_g:.4f}  NIG={ks_n:.4f}  Improvement={imp:.1f}%")

print(f"\n{'='*60}")
if var95_pass == n_assets:
    print("OVERALL: Model performs well at 95% confidence across all assets.")
else:
    print("OVERALL: Some assets show model deficiencies — see limitations.")

if var99_pass < n_assets:
    failed = master_df[master_df["VaR99 result"] != "GREEN"].index.tolist()
    print(f"VaR 99% failures: {failed}")
    print("Consider: GJR-GARCH for leverage effect, longer estimation windows,")
    print("or regime-switching models for these assets.")

MODEL ASSESSMENT VERDICT

VaR 99% exceedance: 4/5 assets pass
VaR 95% exceedance: 5/5 assets pass
KS test (NIG PIT):  5/5 assets pass
Christoffersen 99%: 4/5 independent
Christoffersen 95%: 3/5 independent

KS improvement (NIG vs Gaussian):
  AAPL          Gauss=0.0490  NIG=0.0164  Improvement=66.5%
  EURUSD=X      Gauss=0.0343  NIG=0.0139  Improvement=59.5%
  GLD           Gauss=0.0389  NIG=0.0227  Improvement=41.6%
  TIP           Gauss=0.0368  NIG=0.0295  Improvement=19.8%
  ^GSPC         Gauss=0.0289  NIG=0.0230  Improvement=20.4%

OVERALL: Model performs well at 95% confidence across all assets.
VaR 99% failures: ['^GSPC']
Consider: GJR-GARCH for leverage effect, longer estimation windows,
or regime-switching models for these assets.


---

In [10]:
# Save all assessment results

save_processed(master_df.reset_index(), "master_assessment.parquet")
save_processed(ks_df.reset_index(), "ks_results.parquet")
save_processed(ad_df.reset_index(), "ad_results.parquet")
save_processed(christ_df, "christoffersen_results.parquet")

print("All assessment results saved.")
print("Proceed to notebook 06 (figure export) for publication-quality figures.")

Saved to C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\master_assessment.parquet
Saved to C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\ks_results.parquet
Saved to C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\ad_results.parquet
Saved to C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\christoffersen_results.parquet
All assessment results saved.
Proceed to notebook 06 (figure export) for publication-quality figures.


---